# Pose Extraction on REHAB24-6 (Camera17) — Full Repetition Frames + Normalization

This notebook extracts **MediaPipe Pose landmarks for the full repetition segment** in **REHAB24-6** using the annotated:

- `first_frame`
- `last_frame`

from `Segmentation.csv`.

1. Reads the exact repetition boundaries from `Segmentation.csv`
2. Extracts pose for **every frame from `first_frame` to `last_frame` inclusive**
3. Applies pose normalization per repetition
4. Saves one `.npy` file per repetition with shape:

```python
(T, 33, 4)
```

where:

- `T` = actual repetition length in frames
- `33` = MediaPipe body landmarks
- `4` = `[x, y, z, visibility]`

## Outputs
- Pose clips saved as `.npy`
- `pose_index_full_reps.csv` with metadata + saved pose path + clip length

In [1]:
# 1) Imports
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import mediapipe as mp

## 2) Paths

In [2]:
# 2) Paths
VIDEOS_DIR = Path("../dataset/rehab24-6/videos")
SEGMENTATION_CSV = Path("../dataset/rehab24-6/clean_segmentation.csv")

# Output folder: full repetition clips (all frames)
OUT_DIR = Path("pose_clips/full_reps/camera17")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("VIDEOS_DIR:", VIDEOS_DIR.resolve())
print("SEGMENTATION_CSV:", SEGMENTATION_CSV.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("VIDEOS_DIR exists?", VIDEOS_DIR.exists())
print("SEGMENTATION_CSV exists?", SEGMENTATION_CSV.exists())
print("Sample file exists?", (VIDEOS_DIR / "Ex1" / "PM_000-Camera17-30fps.mp4").exists())

VIDEOS_DIR: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\dataset\rehab24-6\videos
SEGMENTATION_CSV: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\dataset\rehab24-6\clean_segmentation.csv
OUT_DIR: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17
VIDEOS_DIR exists? True
SEGMENTATION_CSV exists? True
Sample file exists? True


## 3) Load segmentation
We read the repetition boundaries directly from the dataset CSV.

In [3]:
# 3) Load segmentation
seg = pd.read_csv(SEGMENTATION_CSV)

print("Segmentation shape:", seg.shape)
print("Columns:", seg.columns.tolist())

required_cols = [
    "video_id", "repetition_number", "exercise_id", "person_id",
    "first_frame", "last_frame"
]
missing_cols = [c for c in required_cols if c not in seg.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in segmentation file: {missing_cols}")

Segmentation shape: (1072, 13)
Columns: ['video_id', 'repetition_number', 'exercise_id', 'person_id', 'first_frame', 'last_frame', 'cam17_orientation', 'mocap_erroneous', 'exercise_subtype', 'lights_on', 'extra_person_in_cam17', 'extra_person_in_cam18', 'correctness']


In [4]:
seg = seg.copy()
seg["first_frame"] = seg["first_frame"].astype(int)
seg["last_frame"]  = seg["last_frame"].astype(int)

seg.head()

,video_id,repetition_number,exercise_id,person_id,first_frame,last_frame,cam17_orientation,mocap_erroneous,exercise_subtype,lights_on,extra_person_in_cam17,extra_person_in_cam18,correctness
0,PM_000,1,1,1,180,377,front,0,right arm,0,3,0,1
1,PM_000,2,1,1,378,620,front,0,right arm,0,3,0,1
2,PM_000,3,1,1,621,865,front,0,right arm,0,3,0,1
3,PM_000,4,1,1,866,1085,front,0,right arm,0,3,3,1
4,PM_000,5,1,1,1086,1265,front,0,right arm,0,3,3,1


## 4) Build absolute video paths (Camera17)

In [5]:
# 4) Build absolute video paths (Camera17)
def build_video_path_camera17(row):
    exercise_folder = f"Ex{int(row['exercise_id'])}"
    filename = f"{row['video_id']}-Camera17-30fps.mp4"
    return str((VIDEOS_DIR / exercise_folder / filename).resolve())

seg["video_path_abs"] = seg.apply(build_video_path_camera17, axis=1)

# Quick existence check (first 10)
print("Video path existence check:")
for i in range(min(10, len(seg))):
    p = Path(seg.loc[i, "video_path_abs"])
    print(i, p.name, "->", p.exists())

# Filter out rows where the video file is missing
missing = ~seg["video_path_abs"].apply(lambda p: Path(p).exists())
if missing.any():
    print(f"WARNING: {missing.sum()} rows have missing video files. They will be skipped.")
    seg = seg[~missing].reset_index(drop=True)

print("Rows after missing-file filter:", len(seg))

Video path existence check:
0 PM_000-Camera17-30fps.mp4 -> True
1 PM_000-Camera17-30fps.mp4 -> True
2 PM_000-Camera17-30fps.mp4 -> True
3 PM_000-Camera17-30fps.mp4 -> True
4 PM_000-Camera17-30fps.mp4 -> True
5 PM_001-Camera17-30fps.mp4 -> True
6 PM_001-Camera17-30fps.mp4 -> True
7 PM_001-Camera17-30fps.mp4 -> True
8 PM_001-Camera17-30fps.mp4 -> True
9 PM_001-Camera17-30fps.mp4 -> True
Rows after missing-file filter: 1072


## 5) Pose normalization

1. Center at pelvis midpoint
2. Scale by torso length

Output shape stays:

```python
(T, 33, 4)
```

In [6]:
# 5) Normalization utilities

LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_HIP, RIGHT_HIP = 23, 24

def normalize_pose_clip_mediapipe(
    clip: np.ndarray,
    eps: float = 1e-6
) -> np.ndarray:
    """
    clip: (T, 33, 4) with [x, y, z, visibility]

    Normalization:
    1. Center at pelvis
    2. Scale by torso length
    """

    clip = np.asarray(clip, dtype=np.float32)

    if clip.ndim != 3 or clip.shape[1] != 33 or clip.shape[2] != 4:
        raise ValueError(f"Expected clip shape (T,33,4), got {clip.shape}")

    xy = clip[..., :2]
    z = clip[..., 2:3]
    vis = clip[..., 3:4]

    # 1) center: pelvis midpoint
    pelvis = (xy[:, LEFT_HIP] + xy[:, RIGHT_HIP]) / 2.0
    xy = xy - pelvis[:, None, :]

    # 2) scale: torso length
    shoulder_mid = (xy[:, LEFT_SHOULDER] + xy[:, RIGHT_SHOULDER]) / 2.0
    torso = np.linalg.norm(shoulder_mid, axis=1)
    torso = np.maximum(torso, eps)

    xy = xy / torso[:, None, None]
    z = z / torso[:, None, None]

    out = np.concatenate([xy, z, vis], axis=-1)
    return out

## 6) MediaPipe pose extraction from the full repetition interval
Etracts **all frames** from `start_f` to `end_f` inclusive.

- iterate over the entire frame range:
```python
range(start_f, end_f + 1)
```

So the repetition length is preserved exactly.

In [7]:
# 6) MediaPipe pose extraction (full repetition frames)

mp_pose = mp.solutions.pose

def extract_pose_clip_full_repetition(
    video_path: Path,
    start_f: int,
    end_f: int,
    det_conf: float = 0.3,
    track_conf: float = 0.3,
    model_complexity: int = 2,
    carry_forward: bool = True,
) -> np.ndarray:
    '''
    Extracts pose from every frame in [start_f, end_f] inclusive.

    Returns
    -------
    pose_clip : np.ndarray
        Shape (T, 33, 4) where T = end_f - start_f + 1
    '''
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    if end_f < start_f:
        return np.empty((0, 33, 4), dtype=np.float32)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError(f"Could not read frame count for: {video_path}")

    start_f = max(0, int(start_f))
    end_f = min(int(end_f), total_frames - 1)

    if end_f < start_f:
        cap.release()
        return np.empty((0, 33, 4), dtype=np.float32)

    frame_indices = list(range(start_f, end_f + 1))
    T = len(frame_indices)

    clip_pose = np.full((T, 33, 4), np.nan, dtype=np.float32)
    last_good = None

    with mp_pose.Pose(
        static_image_mode=False,
        model_complexity=model_complexity,
        smooth_landmarks=True,
        enable_segmentation=False,
        min_detection_confidence=det_conf,
        min_tracking_confidence=track_conf,
    ) as pose:

        for i, frame_idx in enumerate(frame_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            ok, frame_bgr = cap.read()

            if not ok:
                if carry_forward and last_good is not None:
                    clip_pose[i] = last_good
                continue

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            res = pose.process(frame_rgb)

            if res.pose_landmarks is None:
                if carry_forward and last_good is not None:
                    clip_pose[i] = last_good
                continue

            for j, lm in enumerate(res.pose_landmarks.landmark):
                clip_pose[i, j, 0] = lm.x
                clip_pose[i, j, 1] = lm.y
                clip_pose[i, j, 2] = lm.z
                clip_pose[i, j, 3] = lm.visibility

            last_good = clip_pose[i].copy()

    cap.release()

    # normalize after extraction
    clip_pose = normalize_pose_clip_mediapipe(clip_pose)
    return clip_pose

## 7) Extract all repetitions and save clips + index
Each saved file corresponds to **one repetition** and keeps its real length.

### Filename format
```python
{video_id}_r{repetition_number}_f{start}-{end}_pose_full.npy
```

In [9]:
# 7) Extract all repetitions and save clips + index

rows = []
failed = 0

for _, row in tqdm(seg.iterrows(), total=len(seg), desc="Extracting full repetition poses (Camera17)"):
    video_path = Path(row["video_path_abs"])
    start_f = int(row["first_frame"])
    end_f   = int(row["last_frame"])

    try:
        pose_clip = extract_pose_clip_full_repetition(
            video_path=video_path,
            start_f=start_f,
            end_f=end_f,
            det_conf=0.3,
            track_conf=0.3,
            model_complexity=2,
            carry_forward=True,
        )

        out_name = f"{row['video_id']}_r{int(row['repetition_number'])}_f{start_f}-{end_f}_pose_full.npy"
        out_path = (OUT_DIR / out_name).resolve()
        np.save(out_path, pose_clip)

        r = row.to_dict()
        r["pose_path"] = str(out_path)
        r["num_pose_frames"] = int(pose_clip.shape[0])
        r["pose_shape"] = str(tuple(pose_clip.shape))
        rows.append(r)

    except Exception as e:
        failed += 1
        print(f"[FAILED] {row['video_id']} Ex{row['exercise_id']} rep{row['repetition_number']} frames {start_f}-{end_f}: {e}")

pose_index = pd.DataFrame(rows)

POSE_INDEX_PATH = (OUT_DIR / "pose_index_full_reps.csv").resolve()
pose_index.to_csv(POSE_INDEX_PATH, index=False)

print("\nSaved pose index:", POSE_INDEX_PATH)
print("Total saved clips:", len(pose_index))
print("Failed segments:", failed)

if len(pose_index):
    print("\nExample rows:")
    print(pose_index.head())

Extracting full repetition poses (Camera17): 100%|██████████| 1072/1072 [5:27:00<00:00, 18.30s/it] 


Saved pose index: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17\pose_index_full_reps.csv
Total saved clips: 1072
Failed segments: 0

Example rows:
  video_id  repetition_number  exercise_id  person_id  first_frame  \
0   PM_000                  1            1          1          180   
1   PM_000                  2            1          1          378   
2   PM_000                  3            1          1          621   
3   PM_000                  4            1          1          866   
4   PM_000                  5            1          1         1086   

   last_frame cam17_orientation  mocap_erroneous exercise_subtype  lights_on  \
0         377             front                0        right arm          0   
1         620             front                0        right arm          0   
2         865             front                0        right arm          0   
3        1085             front                0        right arm    

## 8) Quick quality checks
We check:

- NaN ratio over `x, y, z`
- saved sequence length stats

In [10]:
# 8) Quick quality checks

def nan_ratio_pose(pose_clip: np.ndarray) -> float:
    xyz = pose_clip[..., :3]
    return float(np.isnan(xyz).sum() / xyz.size) if xyz.size else np.nan

sample_n = min(30, len(pose_index))
ratios = []
lengths = []

for p in pose_index["pose_path"].head(sample_n):
    arr = np.load(p)
    ratios.append(nan_ratio_pose(arr))
    lengths.append(arr.shape[0])

if ratios:
    print("NaN ratio stats on sample:")
    print("  min :", float(np.nanmin(ratios)))
    print("  mean:", float(np.nanmean(ratios)))
    print("  max :", float(np.nanmax(ratios)))
else:
    print("No saved clips to check for NaN ratio.")

if len(pose_index):
    print("\nFull dataset length stats:")
    print("  min frames :", int(pose_index["num_pose_frames"].min()))
    print("  mean frames:", float(pose_index["num_pose_frames"].mean()))
    print("  max frames :", int(pose_index["num_pose_frames"].max()))

NaN ratio stats on sample:
  min : 0.0
  mean: 0.0
  max : 0.0

Full dataset length stats:
  min frames : 15
  mean frames: 119.81623134328358
  max frames : 585


## 9) Inspect one saved clip
Confirm the new extraction keeps the full repetition length.

In [11]:
# 9) Inspect one saved clip
if len(pose_index):
    sample_path = pose_index.loc[0, "pose_path"]
    sample_arr = np.load(sample_path)
    print("Sample path:", sample_path)
    print("Sample shape:", sample_arr.shape)
    print("First frame, first 3 landmarks:")
    print(sample_arr[0, :3, :])
else:
    print("pose_index is empty.")

Sample path: C:\Users\nadaf\Desktop\DS+Code\Rehab24-class\classRehabV2\pose_clips\full_reps\camera17\PM_000_r1_f180-377_pose_full.npy
Sample shape: (198, 33, 4)
First frame, first 3 landmarks:
[[-0.01990113 -1.3990937  -1.4413084   0.99999666]
 [-0.00612071 -1.4550065  -1.3734392   0.99999297]
 [ 0.00947364 -1.4521983  -1.3745269   0.99999106]]
